In [7]:
%load_ext autoreload
%autoreload 2

import sys
import OCR.utils.ovo_svm as ovo_svm

sys.modules["ovo_svm"] = ovo_svm

import gc
from visual.SceneSegmenter import SceneSegmenter
from OCR.src.OCR import OCR

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import os

import cv2


def _run_pipeline(video_path: str,
                   detector: str = 'craft', recognizer: str = 'easyocr'):

    video_id = 404
    json_folder = "./json_outputs"
    os.makedirs("debug/shotsDP", exist_ok=True)
    print("Scene segmentation...")
    segmenter = SceneSegmenter(video_path)
    segmenter.segment_video(threshold=2)
    frames = segmenter.get_frames_with_metadata()
    for frame in frames:
        cv2.imwrite(f"debug/shotsDP/frame_{frame['frame_number']}.jpg", frame['frame'])
    
    print(f"Extracted {len(frames)} frames from video {video_path}")
    del segmenter; gc.collect()
    

    print("OCR processing...")
    ocr = OCR(frames, video_path, video_id, detector=detector, recognizer=recognizer)
    ocr.process_frames()
    ocr_index = ocr.get_inverted_index()
    print(f"OCR inverted index for video {video_id}: {ocr_index}")
    del ocr; gc.collect()
    # update("Object detection...")
    # obj_det = ObjectDetector(video_path, frames)
    # obj_det.detect_objects()
    # ObjectRepository().save_from_inverted_index(obj_det.get_inverted_index(), video_id)
    # del obj_det; gc.collect()
    # update("Visual relationship detection...")
    # vrd = VRD(frames=frames, video_path=video_path, api_key=os.getenv("GEMINI_TOKEN"))
    # vrd.detect_relationships()
    # VRDRepository().save_from_inverted_index(vrd.get_inverted_index(), video_id)
    # del vrd
    # if torch.cuda.is_available():
    #     torch.cuda.empty_cache()
    # gc.collect()
    # update("Audio transcription...")
    # asr = ASR(video_path=video_path, model_name="openai/whisper-small")
    # asr.transcribe(task="translate")
    # transcription_path = os.path.join(json_folder, f"{video_id}_transcription.json")
    # asr.save_transcription(transcription_path)
    # del asr
    # if torch.cuda.is_available():
    #     torch.cuda.empty_cache()
    # gc.collect()
    # update("Sentence segmentation...")
    # seg = SentenceSegmentation(
    #     video_path=video_path,
    #     transcript_json=transcription_path,
    #     similarity_threshold=0.75,
    # )
    # seg.segment()
    # transcript_segments = seg.segments
    # del seg; gc.collect()
    # update("Embedding and storing...")
    # transformer = Transformer(ocr_index, transcript_segments)
    # transformer.transform()
    # hnsw_store = HNSWVectorStore()
    # transformer.save_embeddings(hnsw_store)
    # hnsw_store.commit()  # flush vectors + graph to disk atomically
    print("Done")

In [9]:
path = "./videos/1.mp4"
_run_pipeline(path)

Scene segmentation...
Extracting frames from 23 scenes...
Extracted 46 frames from 23 scenes
Extracted 46 frames from video ./videos/1.mp4
OCR processing...


FileNotFoundError: [Errno 2] No such file or directory: 'g:\\Engineering\\GP\\OCR\\models\\CRAFT_dynamic_5k\\craft_epoch4_dynamic_5k.pth'